# Übung 3: Data Understanding und Regex

Datensatz: `tweet_emotion.parquet` – Twitter-Nachrichten mit zugehöriger Emotions-Klasse.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import re

plt.rcParams["figure.figsize"] = (10, 5)

## 1 Data Understanding

In [ ]:
df = pl.read_parquet("../data/tweet_emotion.parquet")
df.head()

In [ ]:
print(f"Zeilen: {df.shape[0]}, Spalten: {df.shape[1]}")

In [ ]:
print("Spalten:", df.columns)

In [ ]:
df.describe()

### Beobachtungen

- Der Datensatz enthält **40.000 Zeilen** und **3 Spalten**.
- Die Spaltennamen `tweet_id`, `content` und `emotion` sind selbsterklärend.
- `describe()` zeigt 13 einzigartige Emotionsklassen; die häufigste ist `neutral`.
- Es gibt **keine fehlenden Werte** (null_count = 0 in allen Spalten).

## 2 Data Understanding II

In [ ]:
feature_info = pl.DataFrame({
    "Merkmal":      ["tweet_id",                    "content",             "emotion"],
    "Bedeutung":    ["Eindeutige ID des Tweets",     "Rohtext des Tweets",  "Emotionsklasse (Label)"],
    "Skalenniveau": ["Nominal",                      "Nominal (Text)",      "Nominal"],
    "Datentyp":     ["UInt32",                        "String",              "String"],
    "Relevant":     ["Nein (nur Identifikator)",      "Ja – Input",         "Ja – Zielvariable"],
})
feature_info

In [ ]:
print("Schema:", df.schema)
print("Zeilen:", df.shape[0])
print("Spalten:", df.shape[1])
print("Fehlende Werte:")
print(df.null_count())

In [ ]:
# tweet_id ist eine fortlaufende Nummer ohne inhaltliche Bedeutung
df_model = df.drop(["tweet_id"])
print("Verbleibende Spalten:", df_model.columns)

In [ ]:
emotion_counts = df["emotion"].value_counts(sort=True)
labels = emotion_counts["emotion"].to_list()
values = emotion_counts["count"].to_list()

colors = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(labels, values, color=colors[:len(labels)])
ax.set_title("Verteilung der Zielvariable 'emotion'", fontsize=14)
ax.set_xlabel("Emotion")
ax.set_ylabel("Anzahl")
ax.tick_params(axis="x", rotation=45)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 3 Data Understanding – Visualisierungen

In [ ]:
df = df.with_columns(
    pl.col("content").str.len_chars().alias("text_length")
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df["text_length"].to_list(), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Verteilung der Textlänge")
axes[0].set_xlabel("Zeichen")
axes[0].set_ylabel("Anzahl Tweets")

emotions = df["emotion"].unique().sort().to_list()
data_per_emotion = [df.filter(pl.col("emotion") == e)["text_length"].to_list() for e in emotions]
axes[1].boxplot(data_per_emotion, labels=emotions)
axes[1].set_title("Textlänge je Emotion")
axes[1].set_xlabel("Emotion")
axes[1].set_ylabel("Zeichen")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
total = df.shape[0]
pct_labels = [f"{l} ({v/total*100:.1f}%)" for l, v in zip(labels, values)]

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(values, labels=pct_labels, colors=colors[:len(labels)], startangle=140)
ax.set_title("Anteil der Emotionsklassen", fontsize=13)
plt.tight_layout()
plt.show()

### Erkenntnisse

- **Klassenungleichgewicht**: `neutral` (21,6 %) und `worry` (21,1 %) dominieren; `anger` hat nur 110 Einträge (0,3 %).
- Die **Textlänge** variiert kaum zwischen den Emotionsklassen – Emotion ist kein direktes Proxy für Länge.
- Für das spätere Modell muss das Ungleichgewicht berücksichtigt werden (z. B. Class Weights oder Oversampling).

## 4 Data Preparation – Regex

Tweets enthalten häufig Elemente, die für die Emotionsklassifikation irrelevant sind:
URLs, Mentions (`@user`), Hashtags (`#topic`), Encoding-Artefakte und Sonderzeichen.

In [ ]:
def preprocess_tweet(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # URLs entfernen
    text = re.sub(r"@\w+", "", text)                    # Mentions entfernen
    text = re.sub(r"#\w+", "", text)                    # Hashtags entfernen
    text = re.sub(r"[^\w\s]", "", text)                 # Sonderzeichen entfernen
    text = re.sub(r"\d+", "", text)                     # Zahlen entfernen
    text = re.sub(r"\s+", " ", text).strip()            # Leerzeichen normalisieren
    text = text.lower()                                 # Kleinschreibung
    return text

df = df.with_columns(
    pl.col("content").map_elements(preprocess_tweet, return_dtype=pl.String).alias("content_clean")
)
df.select(["content", "content_clean"]).head(10)

In [ ]:
# Vorher/Nachher für Tweets mit URLs, Mentions oder Hashtags
df.filter(
    pl.col("content").str.contains(r"https?://|@|#")
).select(["content", "content_clean"]).head(8)

### Dokumentation der Vorverarbeitungsschritte

| Schritt | Regex-Pattern | Beschreibung |
|---------|---------------|--------------|
| URLs entfernen | `https?://\S+\|www\.\S+` | HTTP/HTTPS-Links und www-Adressen |
| Mentions entfernen | `@\w+` | Twitter-Nutzernamen wie `@user` |
| Hashtags entfernen | `#\w+` | Hashtags wie `#happy` |
| Sonderzeichen entfernen | `[^\w\s]` | Punkte, Kommas, Ausrufezeichen etc. |
| Zahlen entfernen | `\d+` | Reine Ziffernfolgen |
| Leerzeichen normalisieren | `\s+` → `" "` | Mehrfache Leerzeichen auf eines reduzieren |
| Kleinschreibung | `.lower()` | Vereinheitlichung Groß-/Kleinschreibung |